# 06 — Banc d'essai des modèles de lecture manuscrite

**Question.** Parmi les modèles proposés par l'équipe, lesquels lisent le mieux les noms et
prénoms des bulletins, tels quels puis ajustés sur nos données ?



**Protocole, identique pour tous les modèles :**
- les **mêmes 335 lignes** et les **mêmes 5 blocs** que la validation croisée du notebook 05 ;
- **phase 1 — tels quels** : chaque modèle lit toutes les lignes, sans entraînement
  (quelques minutes par modèle) ;
- **phase 2 — ajustés** : seulement les modèles prometteurs en phase 1, en partant de leurs
  poids, testés sur des lignes jamais vues ;
- chaque résultat est **tracé dans MLflow** (étude `banc-htr`, une étiquette par modèle), avec
  les lectures ligne par ligne en pièce jointe.

**Avant la première exécution :** lancer une fois, dans le notebook 05, la cellule
« Export du jeu de lignes pour le banc » : elle prépare `~/work/banc-htr/donnees`.

Le code commun est dans `src/mesure_qualite/perception/banc_htr.py`.

In [ ]:
# 1. Reglages — la liste des modeles est ici : ajouter un modele = ajouter une ligne
import sys
from pathlib import Path

import mlflow
import pandas as pd

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RACINE / "src"))
from mesure_qualite.evaluation import suivi
from mesure_qualite.perception import banc_htr as bh

BANC = Path.home() / "work" / "banc-htr"
DONNEES, MODELES_DIR, TRAVAIL, RESULTATS = (BANC / "donnees", BANC / "modeles",
                                            BANC / "travail", BANC / "resultats")
for d in [MODELES_DIR, TRAVAIL, RESULTATS]:
    d.mkdir(parents=True, exist_ok=True)

MODELES = [
    # PyLaia (Teklia) : modeles de lecture de lignes, du plus proche au plus eloigne de nos donnees
    {"nom": "pylaia-popp",    "type": "pylaia", "depot": "Teklia/pylaia-popp"},     # recensement de Paris, XXe s.
    {"nom": "pylaia-rimes",   "type": "pylaia", "depot": "Teklia/pylaia-rimes"},    # courriers manuscrits modernes
    {"nom": "pylaia-belfort", "type": "pylaia", "depot": "Teklia/pylaia-belfort"},  # conseil municipal, XIXe s.
    {"nom": "pylaia-pellet",  "type": "pylaia", "depot": "Teklia/pylaia-PELLET-Casimir-Marius"},
    {"nom": "pylaia-himanis", "type": "pylaia", "depot": "Teklia/pylaia-himanis"},  # medieval
    # TrOCR (Hugging Face)
    {"nom": "trocr-base",     "type": "trocr",  "depot": "microsoft/trocr-base-handwritten"},
    {"nom": "trocr-large-fr", "type": "trocr",  "depot": "agomberto/trocr-large-handwritten-fr",
     "processeur": "microsoft/trocr-large-handwritten"},
]

LIGNES = bh.charger_lignes(DONNEES)
print(f"{len(LIGNES)} lignes | {LIGNES.cle.nunique()} bulletins | blocs {sorted(int(b) for b in LIGNES.bloc.unique())}")
print("modeles :", ", ".join(m["nom"] for m in MODELES))

---
## Phase 1 — Les modèles tels quels

Chaque modèle lit les 335 lignes. Le résultat est mis en cache dans
`~/work/banc-htr/resultats/brut_<modele>.csv` : une relance ne refait que ce qui manque.
Un modèle qui échoue est **signalé et sauté** : les autres continuent.

In [ ]:
# 2. Phase 1 : lecture par chaque modele, tel quel
import time

BRUT, ECHECS = {}, {}
for m in MODELES:
    cache = RESULTATS / f"brut_{m['nom']}.csv"
    if cache.exists():
        BRUT[m["nom"]] = pd.read_csv(cache, dtype={"attendu": str, "lu": str}).fillna("")
        print(f"{m['nom']:16} relu depuis le cache")
        continue
    print(f"{m['nom']:16} lecture en cours...", end=" ", flush=True)
    t0 = time.time()
    try:
        if m["type"] == "pylaia":
            dossier = bh.pylaia_telecharger(m["depot"], MODELES_DIR / m["nom"])
            lus = bh.PyLaia(dossier, TRAVAIL / m["nom"], hauteur=m.get("hauteur", 128)).lire(LIGNES)
        else:
            lus = bh.trocr_lire(m["depot"], LIGNES, processeur=m.get("processeur"))
    except Exception as e:
        ECHECS[m["nom"]] = f"{type(e).__name__}: {str(e)[:300]}"
        print("ECHEC")
        continue
    mes = bh.mesurer(LIGNES, lus)
    mes.to_csv(cache, index=False)
    BRUT[m["nom"]] = mes
    duree = time.time() - t0
    r = bh.resumer(mes)["TOUS"]
    print(f"exact {r['exact']:.1%} | CER {r['cer']:.1%} | vides {r['vides']:.0%} | {duree / 60:.1f} min")

    with suivi.essai("banc-htr", notebook="06", nom=f"{m['nom']}-brut",
                     modele=m["nom"], depot=m["depot"], phase="brut") as run:
        mlflow.log_params({"depot": m["depot"], "type": m["type"], "lignes": len(mes)})
        mlflow.log_metrics({f"{c}_{k}": float(v) for c, d in bh.resumer(mes).items()
                            for k, v in d.items() if k != "N"} | {"duree_min": duree / 60})
        mlflow.log_artifact(str(cache))

for nom, erreur in ECHECS.items():
    print(f"\nECHEC {nom} :\n  {erreur}")

In [ ]:
# 3. Tableau de  la phase 1
lignes = []
for nom, mes in BRUT.items():
    for champ, r in bh.resumer(mes).items():
        lignes.append({"modele": nom, "champ": champ, "N": r["N"],
                       "exact_%": round(r["exact"] * 100, 1),
                       "IC95": f"[{r['ic_bas'] * 100:.1f} ; {r['ic_haut'] * 100:.1f}]",
                       "CER_%": round(r["cer"] * 100, 1), "vides_%": round(r["vides"] * 100, 1)})
PHASE1 = pd.DataFrame(lignes)
PHASE1.to_csv(RESULTATS / "banc_phase1.csv", index=False)
tous = PHASE1[PHASE1.champ == "TOUS"].sort_values("CER_%")
display(tous)
print("\nLe CER classe les modeles tels quels : plus il est bas, plus le modele 'voit' deja nos"
      "\nlettres, et plus l'ajustement a de chances de marcher. Choisir la phase 2 en consequence.")

---
## Phase 2 — Ajuster les modèles prometteurs (PyLaia)

Mettre dans `AJUSTER` les modèles PyLaia retenus après la phase 1 (CER le plus bas).

- `TEST_RAPIDE = True` : **un seul bloc** (environ 35 minutes par modèle), pour juger vite ;
- `TEST_RAPIDE = False` : **les 5 blocs** (environ 3 heures par modèle), le chiffre définitif.

Réglages appris avec Belfort : 100 époques, pas d'arrêt anticipé (PyLaia reste « muet »
plusieurs dizaines d'époques avant de lire), pas d'apprentissage 5e-4.

**Ajuster un modèle TrOCR** (par exemple `trocr-large-fr`) : dans le notebook 05, remplacer
`MODELE_DEPART` par son dépôt et relancer la validation croisée.

In [ ]:
# 4. Phase 2 : ajustement PyLaia, memes blocs que TrOCR
import numpy as np

AJUSTER = ["pylaia-popp"]     # a choisir d'apres le tableau de la phase 1
TEST_RAPIDE = True
EPOQUES, PAS, PATIENCE, GRAINE = 100, 5e-4, 100, 42

AJUSTE = {}
for nom in AJUSTER:
    m = next(x for x in MODELES if x["nom"] == nom)
    pl = bh.PyLaia(MODELES_DIR / nom, TRAVAIL / nom, hauteur=m.get("hauteur", 128))
    blocs = sorted(LIGNES.bloc.unique())[:1] if TEST_RAPIDE else sorted(LIGNES.bloc.unique())
    resultats, courbes, durees = [], {}, {}
    for b in blocs:
        cache = RESULTATS / f"ajuste_{nom}_bloc{b}.csv"
        if cache.exists():
            resultats.append(pd.read_csv(cache, dtype={"attendu": str, "lu": str}).fillna(""))
            print(f"{nom} bloc {b} : relu depuis le cache")
            continue
        test = LIGNES[LIGNES.bloc == b]
        reste = LIGNES[LIGNES.bloc != b]
        cles = sorted(reste.cle.unique())
        np.random.default_rng(GRAINE).shuffle(cles)
        cles_val = set(cles[:max(1, len(cles) // 10)])        # validation : 10 % des bulletins
        train, val = reste[~reste.cle.isin(cles_val)], reste[reste.cle.isin(cles_val)]
        print(f"{nom} bloc {b} : entrainement {len(train)}, validation {len(val)}, test {len(test)}"
              " — en cours...", flush=True)
        rep, poids, duree, courbe = pl.ajuster(train, val, f"bloc{b}", EPOQUES, PAS, PATIENCE)
        mes = bh.mesurer(test, pl.lire(test, f"test_bloc{b}", archi=rep, poids=poids))
        mes.to_csv(cache, index=False)
        resultats.append(mes)
        courbes[b], durees[b] = courbe, duree / 60
        muettes = next((i for i, v in enumerate(courbe) if v < 1.0), len(courbe))
        r = bh.resumer(mes)["TOUS"]
        print(f"  exact {r['exact']:.1%} | CER {r['cer']:.1%} | vides {r['vides']:.0%} | "
              f"{duree / 60:.0f} min | epoques muettes : {muettes}")
    mes = pd.concat(resultats, ignore_index=True)
    AJUSTE[nom] = mes
    mes.to_csv(RESULTATS / f"ajuste_{nom}.csv", index=False)

    with suivi.essai("banc-htr", notebook="06", nom=f"{nom}-ajuste", modele=nom,
                     phase="ajuste", portee=f"blocs-{'-'.join(map(str, blocs))}") as run:
        mlflow.log_params({"epoques_max": EPOQUES, "pas": PAS, "patience": PATIENCE,
                           "graine": GRAINE, "blocs": len(blocs), "lignes_test": len(mes)})
        mlflow.log_metrics({f"{c}_{k}": float(v) for c, d in bh.resumer(mes).items()
                            for k, v in d.items() if k != "N"}
                           | {"duree_min": float(sum(durees.values()))})
        for b, courbe in courbes.items():
            mlflow.log_text(", ".join(map(str, courbe)), f"courbe_va_cer_bloc{b}.txt")
        mlflow.log_artifact(str(RESULTATS / f"ajuste_{nom}.csv"))

---
## Les cas, ligne par ligne

Pour chaque ligne : l'**attendu** et la lecture de **chaque modèle**, tel quel et ajusté, plus
TrOCR ajusté et gemma4 sur bloc (notebook 05) quand leurs fichiers sont présents.


In [ ]:
# 5. Tableau des cas : attendu et lectures de chaque modele
CAS = LIGNES[["id", "bloc", "champ"]].copy()
CAS["attendu"] = LIGNES.texte.map(bh.normaliser)
for nom, mes in BRUT.items():
    CAS = CAS.merge(mes[["id", "lu"]].rename(columns={"lu": f"{nom}"}), on="id", how="left")
for nom, mes in AJUSTE.items():
    CAS = CAS.merge(mes[["id", "lu"]].rename(columns={"lu": f"{nom}_ajuste"}), on="id", how="left")
if Path("cv_trocr.csv").exists():
    _t = pd.read_csv("cv_trocr.csv", dtype=str).fillna("")
    _t = _t[_t.etat == "apres"][["id", "lu_n"]].rename(columns={"lu_n": "trocr_base_ajuste"})
    CAS = CAS.merge(_t, on="id", how="left")
if Path("predictions_gemma4-bloc.csv").exists():
    _g = pd.read_csv("predictions_gemma4-bloc.csv", dtype=str).fillna("")
    CAS = CAS.merge(_g[["id"]].assign(gemma4_bloc=_g.lu.map(bh.normaliser)), on="id", how="left")
CAS = CAS.fillna("")
CAS.to_csv(RESULTATS / "banc_cas_par_ligne.csv", index=False)

colonnes = [c for c in CAS.columns if c not in ("id", "bloc", "champ", "attendu")]
print("part lue exactement (sur les lignes ou le modele a lu) :")
for c in colonnes:
    lues = CAS[CAS[c] != ""] if c.endswith("_ajuste") else CAS
    print(f"  {c:22} {(lues[c] == lues.attendu).mean():6.1%}   ({len(lues)} lignes)")
with pd.option_context("display.max_rows", 400, "display.max_colwidth", 24):
    display(CAS)

# Phase 2 bis — Ajuster TrOCR (trocr-large-fr)

Même protocole que pour PyLaia : mêmes blocs, validation prise dans l'entraînement (10 % des bulletins), test sur des lignes jamais vues. À chaque époque, le CER de validation s'affiche ; la meilleure époque est gardée.

    TEST_RAPIDE_T = True : un seul bloc, pour juger vite ;
    TEST_RAPIDE_T = False : les 5 blocs, le chiffre comparable à TrOCR-base ajusté du notebook 05 (60,3 %).

Si un message OutOfMemoryError apparaît : passer LOT_T à 2 et relancer.

In [ ]:
# 4 bis. Phase 2 : ajustement TrOCR, memes blocs que le notebook 05
import numpy as np

AJUSTER_TROCR = ["trocr-large-fr"]
TEST_RAPIDE_T = True
EPOQUES_T, PAS_T, LOT_T, GRAINE = 12, 1e-5, 4, 42

AJUSTE = globals().get("AJUSTE", {})
for nom in AJUSTER_TROCR:
    m = next(x for x in MODELES if x["nom"] == nom)
    blocs = sorted(int(b) for b in LIGNES.bloc.unique())
    blocs = blocs[:1] if TEST_RAPIDE_T else blocs
    resultats, courbes, durees = [], {}, {}
    for b in blocs:
        cache = RESULTATS / f"ajuste_{nom}_bloc{b}.csv"
        if cache.exists():
            resultats.append(pd.read_csv(cache, dtype={"attendu": str, "lu": str}).fillna(""))
            print(f"{nom} bloc {b} : relu depuis le cache")
            continue
        test = LIGNES[LIGNES.bloc == b]
        reste = LIGNES[LIGNES.bloc != b]
        cles = sorted(reste.cle.unique())
        np.random.default_rng(GRAINE).shuffle(cles)
        cles_val = set(cles[:max(1, len(cles) // 10)])        # validation : 10 % des bulletins
        train, val = reste[~reste.cle.isin(cles_val)], reste[reste.cle.isin(cles_val)]
        print(f"{nom} bloc {b} : entrainement {len(train)}, validation {len(val)}, test {len(test)}")
        lus, courbe, duree, meilleure = bh.trocr_ajuster(
            m["depot"], train, val, test, processeur=m.get("processeur"),
            epoques=EPOQUES_T, pas=PAS_T, lot=LOT_T, graine=GRAINE)
        mes = bh.mesurer(test, lus)
        mes.to_csv(cache, index=False)
        resultats.append(mes)
        courbes[b], durees[b] = courbe, duree / 60
        r = bh.resumer(mes)["TOUS"]
        print(f"  TEST bloc {b} : exact {r['exact']:.1%} | CER {r['cer']:.1%} | "
              f"meilleure epoque {meilleure} | {duree / 60:.1f} min\n")
    mes = pd.concat(resultats, ignore_index=True)
    AJUSTE[nom] = mes
    mes.to_csv(RESULTATS / f"ajuste_{nom}.csv", index=False)
    r = bh.resumer(mes)
    print(f"{nom} ajuste, {len(mes)} lignes de test :")
    for champ, d in r.items():
        print(f"  {champ:7} exact {d['exact']:.1%} [{d['ic_bas']:.1%} ; {d['ic_haut']:.1%}] | "
              f"CER {d['cer']:.1%}")

    with suivi.essai("banc-htr", notebook="06", nom=f"{nom}-ajuste", modele=nom,
                     phase="ajuste", portee=f"blocs-{'-'.join(map(str, blocs))}") as run:
        mlflow.log_params({"epoques": EPOQUES_T, "pas": PAS_T, "lot": LOT_T, "graine": GRAINE,
                           "blocs": len(blocs), "lignes_test": len(mes), "depot": m["depot"]})
        mlflow.log_metrics({f"{c}_{k}": float(v) for c, d in r.items()
                            for k, v in d.items() if k != "N"}
                           | {"duree_min": float(sum(durees.values()))})
        for b, courbe in courbes.items():
            mlflow.log_text(", ".join(map(str, courbe)), f"courbe_va_cer_bloc{b}.txt")
        mlflow.log_artifact(str(RESULTATS / f"ajuste_{nom}.csv"))
     

In [ ]:
# Recharger les resultats ajustes, refaire le tableau des cas, comparer sur le bloc 1
from scipy.stats import binomtest

AJUSTE = {n: pd.read_csv(RESULTATS / f"ajuste_{n}.csv", dtype={"attendu": str, "lu": str}).fillna("")
          for n in ["trocr-large-fr", "pylaia-popp"] if (RESULTATS / f"ajuste_{n}.csv").exists()}

CAS = LIGNES[["id", "bloc", "champ"]].copy()
CAS["attendu"] = LIGNES.texte.map(bh.normaliser)
for nom, mes in BRUT.items():
    CAS = CAS.merge(mes[["id", "lu"]].rename(columns={"lu": nom}), on="id", how="left")
for nom, mes in AJUSTE.items():
    CAS = CAS.merge(mes[["id", "lu"]].rename(columns={"lu": f"{nom}_ajuste"}), on="id", how="left")
_t = pd.read_csv("cv_trocr.csv", dtype=str).fillna("")
CAS = CAS.merge(_t[_t.etat == "apres"][["id", "lu_n"]].rename(columns={"lu_n": "trocr_base_ajuste"}),
                on="id", how="left")
_g = pd.read_csv("predictions_gemma4-bloc.csv", dtype=str).fillna("")
CAS = CAS.merge(_g[["id"]].assign(gemma4_bloc=_g.lu.map(bh.normaliser)), on="id", how="left").fillna("")
CAS.to_csv(RESULTATS / "banc_cas_par_ligne.csv", index=False)

b1 = CAS[CAS.bloc == 1]
print(f"Bloc 1 : {len(b1)} lignes, memes lignes pour tous les modeles")
for c in ["trocr_base_ajuste", "trocr-large-fr_ajuste", "pylaia-popp_ajuste", "gemma4_bloc"]:
    if c in b1:
        print(f"  {c:24} {(b1[c] == b1.attendu).mean():6.1%}")

a = b1["trocr-large-fr_ajuste"] == b1.attendu
b = b1["trocr_base_ajuste"] == b1.attendu
n10, n01 = int((a & ~b).sum()), int((~a & b).sum())
p = binomtest(n10, n10 + n01).pvalue if n10 + n01 else 1.0
print(f"\nlarge-fr juste seul : {n10} | base juste seul : {n01} | McNemar p = {p:.3f}")
     